# 05 · BGE-small student: alignment + contrastive refinement, full fine-tune (DDP, 2 GPUs)
A second, much smaller student: `BAAI/bge-small-en-v1.5` (BERT, 12 layers, hidden 384, 33M params), CLS pooling,
right padding. A linear head 384→256 maps it into the teacher index space. **Every weight trains** (no LoRA, no
pruning), in two stages:
1. **Align**: put the student's query vector on the teacher's vector of the same query,
   L2 = ||s − t||² = 2 − 2·cos (as in `02`).
2. **Contrastive refinement**: from stage 1, rank the labelled positive above the batch's other teacher docs
   (as in `03`): L = CE(logits / tau, own positive) + l2_weight · L2. Documents stay the frozen teacher `corpus.npy`.

Each stage is its own checkpoint dir (`bge_aligned/`, `bge_aligned_cr/`, each with `final/`) and its own row.
Re-running resumes and a finished stage is skipped.

Input: `01`. Output (attach to `04`): `runs/<mode>/{bge_aligned, bge_aligned_cr}/`, `runs/<mode>/metrics.json`.

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import os
import shutil
import torch
from model import get_device
from teacher import TEACHER_DEFAULTS as T

DEBUG = True    # use the same value as in 01
NPROCS = 2

# --- paths. On Kaggle, earlier notebooks' outputs are attached as inputs (read only); set the slugs to yours.
#     Off Kaggle everything lives under the repo, so every IN_* is the same dir.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/notebooks/tensorpanda231"
IN_01 = f"{KAGGLE_IN}/hlm-01-teacher-embeddings" if ON_KAGGLE else REPO
DATA_PATH = f"{IN_01}/cache/{MODE}/data.json"
EMB_DIR = f"{IN_01}/cache/{MODE}/emb/teacher"
RUN_DIR = f"{ROOT}/runs/{MODE}"
PREV_METRICS = f"{IN_01}/runs/{MODE}/metrics.json"

STUDENT = "BAAI/bge-small-en-v1.5"
# BGE's optional retrieval instruction is "Represent this sentence for searching relevant passages: "; off by
# default to match the other students. Must be the same in ALIGN, REFINE and EVAL.
QUERY_PREFIX = ""

# --- stage 1: alignment, full fine-tune (align.ALIGN_DEFAULTS; unknown keys raise)
ALIGN = dict(model=STUDENT,
             full=True,                           # every base weight + head trains; no LoRA (lora_* unused)
             dim=T["dim"],                        # head output = teacher index dim
             query_prefix=QUERY_PREFIX, max_query_len=32,
             loss="l2", loss_kwargs={},           # ||s - t||^2 = 2 - 2 cos between unit vectors
             lr=5e-5,                             # all base weights; the usual full-FT range for a 33M BERT is 2e-5..1e-4
             head_lr=1e-3,                        # the head starts random
             weight_decay=0.0, warmup_ratio=0.05,
             epochs=1, batch_size=256,            # per GPU; global batch = batch_size * NPROCS
             grad_clip=1.0, amp=True, seed=0,
             log_every=20, eval_every=200, save_every=500, keep_last=2,
             data_path=DATA_PATH, teacher_emb_dir=EMB_DIR, out_dir=f"{RUN_DIR}/bge_aligned")

# --- stage 2: contrastive refinement of every weight (refine.REFINE_DEFAULTS; init_adapter / out_dir set below)
REFINE = dict(dim=T["dim"], query_prefix=QUERY_PREFIX, max_query_len=32,
              tau=0.05,                            # temperature
              margin=0.1,                          # mask negatives with s > s(q, d+) + margin (likely false negatives)
              n_neg=1,                             # negatives per query: 1 = stored one, k > 1 adds k-1 teacher-mined
              l2_weight=0.1,                       # keeps the student near the teacher query space; 0 = pure L_CR
              lr=2e-5, head_lr=1e-4, weight_decay=0.0, warmup_ratio=0.05,
              epochs=1, batch_size=128,            # queries per GPU; docs per step = (1 + n_neg) * batch_size * NPROCS
              grad_clip=1.0, amp=True, seed=0,
              log_every=20, eval_every=200, save_every=500, keep_last=2,
              data_path=DATA_PATH, teacher_emb_dir=EMB_DIR)

# --- evaluation (evaluation.evaluate_model kwargs): student queries vs the frozen teacher corpus, top 100
EVAL = dict(splits=("val", "test"), dim=T["dim"], max_query_len=32, query_prefix=QUERY_PREFIX, device=get_device(),
            dtype="fp16")
LATENCY = dict(n_queries=2000, batch_sizes=(1, 32))

if DEBUG:
    for cfg in (ALIGN, REFINE):
        cfg.update(log_every=5, eval_every=20, save_every=20)
    LATENCY.update(n_queries=200)

## 1 · Check inputs
Teacher vectors valid (finite, unit norm, `data.json` order) for the corpus, val/test and the alignment queries.

In [ ]:
from data import load_emb, load_json
from evaluation import check_emb, load_index

# rows accumulate across notebooks: start from the previous notebook's metrics.json (copied once)
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(f"{RUN_DIR}/metrics.json") and os.path.exists(PREV_METRICS):
    shutil.copy(PREV_METRICS, f"{RUN_DIR}/metrics.json")
data = load_json(DATA_PATH)
index = load_index(data, EMB_DIR, ("val", "test"))   # (teacher docs, {split: teacher queries})
emb, ids = load_emb(EMB_DIR, "align")
check_emb(emb, ids, data["align"]["ids"], "align")
assert set(data["train"]["ids"]) <= set(data["align"]["ids"]), "train queries need teacher vectors (align set)"
print(f"align {emb.shape}: {len(ids) // (ALIGN['batch_size'] * NPROCS)} align steps, "
      f"{len(data['train']['ids']) // (REFINE['batch_size'] * NPROCS)} refine steps")

## 2 · Align (full fine-tune)
Rank r takes its slice of every global batch; DDP averages the gradients of every weight. Val L2 / cos / nDCG@10 are
logged every `eval_every` steps.

In [ ]:
from align import align_main
from run_exp import train_once

runs = {"bge_aligned": train_once(align_main, ALIGN, NPROCS)}

## 3 · Contrastive refinement from the aligned student, compare before / after
`adapter_config.json` has `full: true`, so refinement trains every weight. Trains `bge_aligned_cr` if missing,
evaluates the missing rows (val + test), prints the gain and the table.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from run_exp import exp_cr

res = exp_cr(REFINE, runs, data, index, RUN_DIR, EVAL, NPROCS)
runs["bge_aligned_cr"] = f"{RUN_DIR}/bge_aligned_cr/final"
for fig in res["figs"]:
    display(fig)
plt.close("all")

## 4 · Training curves
`val_loss` is L2 in both stages (comparable); `train_loss` is L2 in stage 1, CR + l2_weight · L2 in stage 2.

In [ ]:
from run_exp import plot_curves, save_fig

fig = plot_curves({n: f"{RUN_DIR}/{n}" for n in runs}, ("train_loss", "val_loss", "val_ndcg@10"), "bge-small")
display(save_fig(fig, RUN_DIR, "bge_small_curves"))
plt.close("all")

## 5 · Retrieval quality (val + test)
Student queries vs the frozen teacher corpus. retention = nDCG@10 / teacher nDCG@10.

In [ ]:
from run_exp import print_rows

metrics = load_json(f"{RUN_DIR}/metrics.json")
for split in ("val", "test"):
    print_rows(metrics, list(runs), split)
    print()

## 6 · Query latency (one GPU, fp16)

In [ ]:
from evaluation import measure_latency
from model import load_encoder
from teacher import free_gpu_memory

texts = data["val"]["texts"][:LATENCY["n_queries"]]
for name, path in runs.items():
    model, tok = load_encoder(path, EVAL["device"], torch.float16)
    print(f"--- {name}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
    measure_latency(model, tok, texts, EVAL["device"], T["dim"], 32, QUERY_PREFIX, LATENCY["batch_sizes"])
    del model
    free_gpu_memory()